# 🧪 Lab 11 — LLM API & Orkestrasi: Sistem yang Tahan Gagal

> Pendamping materi Bab 11. Kemampuan yang dilatih di sini **bukan** "bisa
> memanggil API", tapi **membungkus panggilan itu menjadi sistem**: retry yang
> tahu kapan berhenti, fallback yang sadar harga, cache yang tidak membayar dua
> kali, tool yang divalidasi, dan pipeline yang bisa diaudit per langkah.

Provider di lab ini **palsu dan deterministik** (`project-llmkit/llmkit/transport.py`)
sehingga kegagalan bisa **dijadwalkan**: `rate_limit` pada panggilan ke-1,
`server_error` pada ke-2, pulih di ke-3. Itulah satu-satunya cara melatih retry
tanpa menunggu jam dinding dan tanpa membakar token.

| Bagian | Topik | Waktu (menit) |
|---|---|---|
| 0 | Setup: provider palsu, tarif, rencana kegagalan | 10 |
| 1 | Token, biaya, penjaga anggaran | 20 |
| 2 | Retry + backoff: transien vs fatal | 25 |
| 3 | Fallback chain multi-provider | 20 |
| 4 | Cache konten-address + TTL | 20 |
| 5 | Streaming & TTFT | 15 |
| 6 | Function calling: validasi argumen + batas iterasi | 25 |
| 7 | Orkestrasi pipeline & laporan per langkah | 25 |
| 8 | Ringkasan & pertanyaan | 10 |

Setiap latihan punya cell **✅ Cek** — jalankan; kalau ✅ berarti pemahamanmu benar.
Semua angka terkunci (tanpa jaringan, tanpa API key).


## Bagian 0 — Setup: Provider Palsu yang Bisa Disuruh Gagal

`project-llmkit/llmkit/transport.py` sudah menyediakan `ProviderPalsu` (GIVEN):

- menjawab deterministik dari prompt — tanpa jaringan, tanpa API key;
- gagal sesuai `data.RENCANA`: `mini=['ok']`, `sedang=['rate_limit','ok']`,
  `besar=['rate_limit','server_error','ok']`;
- melaporkan `tokens_in`, `tokens_out`, `latency_ms`, `tool_calls`.

Perhatikan: **jumlah panggilan per model** adalah jam kegagalannya. `provider.panggil`
yang ke-1 untuk `besar` selalu `rate_limit`, yang ke-2 `server_error`, ke-3 `ok`.
Karena itu setiap eksperimen harus memakai **provider baru** kalau ingin mengulang
urutan yang sama.


In [ ]:
import hashlib
import json
import math
import sys
from pathlib import Path

# Cari folder project-llmkit (transport GIVEN ada di sana)
_akar = Path.cwd()
while _akar != _akar.parent and not (_akar / 'project-llmkit' / 'llmkit').is_dir():
    _akar = _akar.parent
sys.path.insert(0, str(_akar / 'project-llmkit'))

from llmkit.data import (HARGA, HARGA_CADANGAN, LANGKAH_PIPELINE, RANTAI_FALLBACK,
                         RENCANA, TIKET_CONTOH, TOOLS)
from llmkit.transport import (GalatAPI, GalatPermintaanBuruk, JENIS_TRANSIEN,
                              ProviderPalsu, provider_cadangan, provider_utama)

PESAN = [{'role': 'user', 'content': 'Jelaskan apa itu machine learning dalam 2 kalimat.'}]

print('transport GIVEN dari:', _akar / 'project-llmkit')
print('tarif USD / 1 juta token:')
for m, t in HARGA.items():
    print('   %-7s input %.2f | output %.2f' % (m, t['input'], t['output']))
print('rencana kegagalan:', RENCANA)
print('jenis galat transien:', JENIS_TRANSIEN)


In [ ]:
# Contoh: menyuruh provider gagal dua kali lalu pulih — tanpa jaringan
p_contoh = provider_utama()
for i in range(3):
    try:
        r = p_contoh.panggil('besar', PESAN)
        print(f'panggilan {i + 1}: OK -> {r["teks"]}')
    except GalatAPI as e:
        print(f'panggilan {i + 1}: {e.jenis} ({e.pesan})')
print('log provider:', [(c['model'], c['status'], c['percobaan']) for c in p_contoh.log])


## Bagian 1 — Token, Biaya, Penjaga Anggaran

Dua angka yang harus kamu pegang **sebelum** menulis prompt: berapa token, dan
berapa dolar. Kalau keduanya tidak bisa dihitung sebelum panggilan, sistemmu
tidak bisa dianggarkan — dan tagihan datang sebagai kejutan.

Konvensi lab ini (sama dengan `project-llmkit/llmkit/cost.py`):

- `token_estimasi(teks) = ceil(len(teks) / 4)`
- tarif di `data.HARGA` adalah **USD per 1 juta token**
- `jalankan_anggaran` berhenti **memanggil** saat anggaran habis; sisanya dilewati


In [ ]:
def token_estimasi(teks):
    """Perkiraan token: ceil(len(teks) / 4)."""
    # TODO
    raise NotImplementedError


def hitung_biaya(tokens_in, tokens_out, model, harga=None):
    """Biaya USD dari tarif per 1 juta token. Model tak dikenal -> ValueError."""
    # TODO
    raise NotImplementedError


In [ ]:
def jalankan_anggaran(provider, daftar_pesan, batas_usd, model='mini', timeout_ms=5000):
    """Proses pekerjaan sampai anggaran HABIS; sisanya DILEWATI (bukan dipaksa).

    Return dict: diproses, dilewati, total_biaya_usd, hasil (list), berhenti_di.
    """
    # TODO
    raise NotImplementedError


In [ ]:
# ✅ Cek latihan 1.1 — token
assert token_estimasi('') == 0
assert [token_estimasi('x' * n) for n in (1, 4, 5, 9)] == [1, 1, 2, 3]
print('✅ Latihan 1.1 benar! Estimasi 4 karakter/token cukup untuk ANGGARAN (bukan untuk tagihan).')


In [ ]:
# ✅ Cek latihan 1.2 — biaya & rasio model
b_mini = hitung_biaya(1000, 300, 'mini')
b_besar = hitung_biaya(1000, 300, 'besar')
print('1k in / 300 out -> mini $%.6f | besar $%.6f | rasio %.1fx' % (b_mini, b_besar, b_besar / b_mini))
assert b_mini == 0.00033
assert b_besar == 0.0095
try:
    hitung_biaya(10, 10, 'model_ajaib')
    raise AssertionError('model tak dikenal harus ValueError')
except ValueError:
    pass
print('✅ Latihan 1.2 benar! Model besar != "lebih baik"; ia ~29x lebih mahal.')


In [ ]:
# ✅ Cek latihan 1.3 — anggaran yang berhenti, bukan bablas
anggaran = jalankan_anggaran(provider_utama(), [PESAN] * 4, 0.00003, model='mini')
print('diproses %d | dilewati %d | berhenti_di %s | total $%.8f'
      % (anggaran['diproses'], anggaran['dilewati'], anggaran['berhenti_di'],
         anggaran['total_biaya_usd']))
assert (anggaran['diproses'], anggaran['dilewati'], anggaran['berhenti_di']) == (3, 1, 3)
assert abs(anggaran['total_biaya_usd'] - 4.005e-05) < 1e-12
print('✅ Latihan 1.3 benar! Batas $0.00003 tercapai di pekerjaan ke-3; ke-4 DILEWATI.')
print('   "Lewati" itu fitur produk: lebih baik 3 hasil + 1 pemberitahuan daripada tagihan bablas.')


### Mikro-eksperimen

- Naikkan anggaran jadi `0.0001` untuk 4 pekerjaan. Berapa yang diproses sekarang?
- Ganti `model='besar'`. Berapa pekerjaan yang tertampung sebelum anggaran habis?

> **Koneksi:** angka per pekerjaan di sini = `biaya_hasil(hasil)`. Di produksi, ini
> yang dicatat per *tenant* + alert saat 80% anggaran harian terpakai.


## Bagian 2 — Retry & Backoff: Transien vs Fatal

Retry bukan "loop sampai berhasil". Ia **dua keputusan**:

1. **Apakah galat ini bisa membaik?** `rate_limit`, `server_error`, `timeout` ya
   (transien). `bad_request` / model tidak ada **tidak** — mengulang hanya membayar
   kegagalan yang sama dua kali.
2. **Berapa lama menunggu?** Backoff eksponensial `basis * faktor^(n-1)`, dibatasi
   `maks_ms`, dan **jangan menunggu setelah percobaan terakhir**.

`sleep_fn` disuntik (bukan `time.sleep`) supaya bisa diuji tanpa menunggu detik nyata.


In [ ]:
class GalatSemuaPercobaan(GalatAPI):
    """Semua percobaan gagal. `riwayat` = daftar (percobaan, jenis)."""

    def __init__(self, model, riwayat, pesan='semua percobaan gagal'):
        super().__init__('semua_gagal', model, pesan)
        self.riwayat = list(riwayat)


In [ ]:
def backoff_ms(percobaan, basis_ms=1000, faktor=2, maks_ms=8000):
    """Jeda sebelum percobaan ke-n: basis * faktor^(n-1), dibatasi maks_ms."""
    # TODO
    raise NotImplementedError


def panggil_dengan_retry(provider, model, messages, maks_percobaan=4,
                         basis_ms=1000, faktor=2, maks_ms=8000,
                         timeout_ms=5000, sleep_fn=None, **params):
    """Panggil provider; ulangi HANYA galat transien, dengan backoff eksponensial.

    Hasil yang sukses ditambah kunci: attempts, timpenungguan_ms, total_ms, riwayat.
    Galat fatal (bad_request/model tak dikenal) langsung dilempar (TIDAK diulang).
    """
    # TODO
    raise NotImplementedError


In [ ]:
# ✅ Cek latihan 2.1 — sukses langsung vs sukses di percobaan ke-2
tunggu = []
h_mini = panggil_dengan_retry(provider_utama(), 'mini', PESAN)
h_sedang = panggil_dengan_retry(provider_utama(), 'sedang', PESAN, sleep_fn=tunggu.append)
print('mini  : attempts', h_mini['attempts'], '| tunggu', h_mini['timpenungguan_ms'])
print('sedang: attempts', h_sedang['attempts'], '| tunggu', h_sedang['timpenungguan_ms'],
      '| riwayat', h_sedang['riwayat'])
print('sedang: latency', h_sedang['latency_ms'], 'ms + tunggu -> total_ms', h_sedang['total_ms'])
assert h_mini['attempts'] == 1 and h_mini['timpenungguan_ms'] == []
assert h_sedang['attempts'] == 2 and tunggu == [1000]
assert h_sedang['total_ms'] == h_sedang['latency_ms'] + 1000 == 1240
print('✅ Latihan 2.1 benar! Kegagalan transien TIDAK gratis: +1000 ms ke latency pengguna.')


In [ ]:
# ✅ Cek latihan 2.2 — percobaan habis vs galat fatal
try:
    panggil_dengan_retry(provider_utama(), 'besar', PESAN, maks_percobaan=2)
    raise AssertionError('harus GalatSemuaPercobaan')
except GalatSemuaPercobaan as e:
    print('percobaan habis -> riwayat', e.riwayat)
    assert e.riwayat == [(1, 'rate_limit'), (2, 'server_error')]

p_fatal = provider_utama()
try:
    panggil_dengan_retry(p_fatal, 'model_ajaib', PESAN, sleep_fn=lambda ms: None)
    raise AssertionError('harus GalatPermintaanBuruk')
except GalatPermintaanBuruk as e:
    print('fatal ->', e.jenis, '| panggilan ke provider:', p_fatal.jumlah_panggilan['model_ajaib'])
    assert p_fatal.jumlah_panggilan['model_ajaib'] == 1, 'fatal TIDAK boleh diulang'
print('✅ Latihan 2.2 benar! `bad_request` dilempar di percobaan ke-1 — nol biaya sia-sia.')


In [ ]:
# ✅ Cek latihan 2.3 — jadwal backoff
peta = [backoff_ms(n) for n in range(1, 6)]
print('peta backoff      :', peta)
print('basis 100, f=3, maks 500:', [backoff_ms(n, 100, 3, 500) for n in range(1, 5)])
assert peta == [1000, 2000, 4000, 8000, 8000]
assert [backoff_ms(n, 100, 3, 500) for n in range(1, 5)] == [100, 300, 500, 500]
print('✅ Latihan 2.3 benar! Naik eksponensial, lalu DIBATASI — tanpa batas, percobaan')
print('   ke-10 akan menunggu lebih dari 8 menit.')


### Mikro-eksperimen

- Gurat `sleep_fn=None` (biarkan `time.sleep` nyata) di `h_sedang` — berapa detik
  latensi nyata yang ditambahkan retry? Kaitkan dengan SLA pengguna.
- Tambahkan **jitter** deterministik: `jeda + rng.randint(0, jeda // 4)`. Kenapa
  produksi nyata butuh ini? (petunjuk: thundering herd)

> **Koneksi:** galat transien inilah yang di Bab 14 dipakai ulang saat agent memanggil
> tool eksternal — dengan tambahan pikiran: **tool call juga punya efek samping**, jadi
> retry tool tulis-data butuh idempotency key.


## Bagian 3 — Fallback Chain: Kesempatan Kedua yang Sadar Harga

Rantai fallback = daftar bernomor, dicoba **berurutan**, tiap tautan boleh punya
`settings` sendiri (mis. "coba keras di model utama, coba cepat di cadangan").

```
RANTAI_FALLBACK = [
  {'provider': 'utama',    'model': 'besar'},   ← rencana: rate_limit, server_error, ok
  {'provider': 'utama',    'model': 'mini'},    ← rencana: ok
  {'provider': 'cadangan', 'model': 'murah'},   ← rencana: ok
]
```

Yang harus kamu hasilkan: **tautan mana yang menang** (`lompatan`), dan **apa yang
sudah dicoba** (`dicoba`) — supaya keputusan turun-kualitas bisa diaudit.


In [ ]:
def panggil_dengan_fallback(providers, rantai, messages, settings=None, **params):
    """Coba tiap tautan rantai (berurutan) sampai satu berhasil.

    `providers`: {'utama': provider, 'cadangan': provider}
    Return hasil panggilan + kunci: dicoba, lompatan, tautan.
    Semua gagal -> GalatSemuaPercobaan(riwayat=...).
    """
    # TODO
    raise NotImplementedError


In [ ]:
# ✅ Cek latihan 3.1 — rantai menyelamatkan permintaan
provs = {'utama': provider_utama(), 'cadangan': provider_cadangan()}
h_fb = panggil_dengan_fallback(provs, RANTAI_FALLBACK, PESAN,
                               settings={'maks_percobaan': 2})
for d in h_fb['dicoba']:
    print('  coba %-8s %-6s sukses=%-5s jenis=%s'
          % (d['provider'], d['model'], d['sukses'], d['jenis']))
print('menang di tautan ke-', h_fb['lompatan'], '->', h_fb['provider'], h_fb['model'])
assert h_fb['lompatan'] == 1 and h_fb['model'] == 'mini'
assert [(d['provider'], d['model'], d['sukses'], d['jenis']) for d in h_fb['dicoba']] == \
       [('utama', 'besar', False, 'semua_gagal'), ('utama', 'mini', True, None)]
print('✅ Latihan 3.1 benar! `besar` gagal DUA kali (rate_limit + server_error) lalu rantai turun.')


In [ ]:
# ✅ Cek latihan 3.2 — apa HARGANYA turun kualitas?
p_murah = provider_utama()
h_mini = panggil_dengan_retry(p_murah, 'mini', PESAN)
p_besar = provider_utama()
h_besar = panggil_dengan_retry(p_besar, 'besar', PESAN, sleep_fn=lambda ms: None)
print('besar sukses di percobaan ke-', h_besar['attempts'])
b_mini = hitung_biaya(h_mini['tokens_in'], h_mini['tokens_out'], 'mini')
b_besar = hitung_biaya(h_besar['tokens_in'], h_besar['tokens_out'], 'besar')
print('model menang :', h_fb['model'], '(kualitas lebih rendah dari besar)')
print('biaya besar  : $%.6f | mini: $%.6f | rasio %.1fx' % (b_besar, b_mini, b_besar / b_mini))
assert b_besar > 10 * b_mini
print('✅ Latihan 3.2 benar! Fallback itu trade-off eksplisit: ketersediaan ⇄ kualitas ⇄ biaya.')
print('   Untuk ekstraksi JSON ber-skema ketat, turun ke model kecil justru bisa lebih mahal:')
print('   hasilnya tampak sukses tapi tidak lolos skema -> retry -> biaya dua kali.')


## Bagian 4 — Cache Konten-Address: Jangan Bayar Dua Kali

Cache prompt di-key oleh **semua yang mengubah output**: model + pesan + parameter
generasi. Kalau salah satu tidak masuk kunci, kamu menyajikan jawaban yang salah
dengan percaya diri.

TTL di sini memakai **tick logis** (bukan jam dinding) supaya kedaluwarsa bisa
diuji deterministik — pola yang sama dipakai di test produksi.

Aturan hitungannya sengaja: **kedaluwarsa dihitung `expired`, bukan `miss`**
(supaya kamu bisa membedakan "cache dingin" dari "cache basi").


In [ ]:
class CachePrompt:
    """Cache konten-address dengan TTL dalam satuan tick logis."""

    def __init__(self, ttl=3):
        # TODO
        raise NotImplementedError

    def kunci(self, model, messages, params=None):
        """SHA-256 (16 hex) dari (model, messages, params) yang diurutkan."""
        # TODO
        raise NotImplementedError

    def ambil(self, kunci, sekarang):
        """Hasil cache bila masih segar; None bila miss ATAU kedaluwarsa."""
        # TODO
        raise NotImplementedError

    def simpan(self, kunci, hasil, sekarang):
        # TODO
        raise NotImplementedError

    def statistik(self):
        """hit, miss, expired, ukuran, hit_rate (hit_rate = 0.0 bila kosong)."""
        # TODO
        raise NotImplementedError


In [ ]:
def panggil_bercache(provider, model, messages, cache, sekarang, **params):
    """Panggil provider, LEWATI kalau cache masih segar. Hasil punya kunci 'cache'
    bernilai 'hit' atau 'miss'."""
    # TODO
    raise NotImplementedError


In [ ]:
# ✅ Cek latihan 4.1 — pola hit/miss/expired
c = CachePrompt(ttl=3)
p_cache = provider_utama()
pola = [panggil_bercache(p_cache, 'mini', PESAN, c, sekarang=t)['cache']
        for t in (0, 1, 5, 6)]
print('pola (tick 0,1,5,6):', pola)
print('statistik          :', c.statistik())
print('panggilan provider :', p_cache.jumlah_panggilan['mini'], '(bukan 4!)')
assert pola == ['miss', 'hit', 'miss', 'hit']
assert p_cache.jumlah_panggilan['mini'] == 2
assert c.statistik()['hit_rate'] == 0.5
assert (c.statistik()['hit'], c.statistik()['miss'], c.statistik()['expired']) == (2, 1, 1)
print('✅ Latihan 4.1 benar! 4 permintaan -> 2 panggilan provider. Tick 5 BUKAN miss:')
print('   ia expired (umur 5 >= ttl 3), itulah sebabnya expired dihitung terpisah.')


In [ ]:
# ✅ Cek latihan 4.2 — kunci cache harus lengkap
k1 = c.kunci('mini', PESAN)
print('kunci mini             :', k1)
print('beda model             :', k1 != c.kunci('sedang', PESAN))
print('beda temperature       :', c.kunci('mini', PESAN, {'temperature': 0.0})
      != c.kunci('mini', PESAN, {'temperature': 1.0}))
print('urutan kunci stabil    :', c.kunci('mini', PESAN) == c.kunci('mini', list(PESAN)))
assert k1 == 'f833386edc2ecf9c'
assert c.kunci('mini', PESAN) != c.kunci('sedang', PESAN)
assert c.kunci('mini', PESAN, {'temperature': 0.0}) != c.kunci('mini', PESAN, {'temperature': 1.0})
print('✅ Latihan 4.2 benar! Di produksi, tambahkan ke kunci: VERSI PROMPT, konteks')
print('   retrieval (Bab 12), snapshot model, dan tenant id untuk data sensitif.')


## Bagian 5 — Streaming: TTFT vs Total

Streaming memperbaiki **persepsi**, bukan total waktu. Ukurannya:

- **TTFT** (*time to first token*) — seberapa cepat pengguna melihat kata pertama;
- **total_ms** — sampai respons selesai (biaya/throughput).

Provider palsu menghasilkan chunk deterministik: chunk pertama pada `TTFT_MS=120`,
selanjutnya `MS_PER_CHUNK=20` per kata, ditutup satu chunk kosong sebagai penanda akhir.


In [ ]:
def konsumsi_stream(chunks):
    """Rakit chunk {'delta', 't_ms'} -> teks + ukuran streaming.

    Return: teks, jumlah_chunk (delta NON-KOSONG), ttft_ms (None bila tak ada kata),
    total_ms (t_ms chunk TERAKHIR).
    """
    # TODO
    raise NotImplementedError


In [ ]:
# ✅ Cek latihan 5.1 — mengukur aliran
s = konsumsi_stream(provider_utama().stream('mini', PESAN))
print('ttft :', s['ttft_ms'], 'ms | total:', s['total_ms'], 'ms | chunk:', s['jumlah_chunk'])
print('teks :', s['teks'])
assert (s['ttft_ms'], s['total_ms'], s['jumlah_chunk']) == (120, 340, 11)
assert s['teks'] == '[mini] jawaban untuk "Jelaskan apa itu machine learning dalam 2 kalimat."'
print('pengguna melihat kata pertama %.0f%% lebih cepat dari total respons'
      % (100 * (1 - s['ttft_ms'] / s['total_ms'])))
assert konsumsi_stream([{'delta': '', 't_ms': 42}])['ttft_ms'] is None
assert konsumsi_stream([{'delta': '', 't_ms': 42}])['teks'] == ''
print('✅ Latihan 5.1 benar! Streaming memindahkan waktu tunggu, TIDAK menghapusnya.')
print('   Tepat untuk chat/CLI & output panjang; sia-sia untuk JSON pendek yang baru')
print('   berguna setelah utuh, atau untuk batch/async yang tidak ditunggu manusia.')


## Bagian 6 — Function Calling: Model Memilih, Kode Mengeksekusi

Prinsip yang tidak boleh dilanggar: **model hanya mengusulkan**. Validasi, izin,
dan eksekusi ada di kodemu.

```
model  ->  tool_calls: [{id, nama, argumen}]     (USULAN, data tak tepercaya)
kode   ->  validasi_argumen -> jalankan_tool     (KEPUTUSAN)
kode   ->  {role: 'tool', ...}                   (OBSERVASI, dikirim balik)
```

Tiga nama tool di `data.TOOLS`: `get_cuaca`, `hitung`, `cari_catatan`.


In [ ]:
def skema_parameter(nama, tools=TOOLS):
    """JSON-schema mini dari contoh argumen: {type, properties, required}."""
    # TODO
    raise NotImplementedError


def skema_tools(tools=TOOLS):
    """Bentuk tools untuk API function calling (list of {type, function})."""
    # TODO
    raise NotImplementedError


In [ ]:
def validasi_argumen(nama, argumen, tools=TOOLS):
    """Daftar error: 'missing:<k>', 'type:<k>', 'tak_dikenal:<k>'.
    Nama tool asing -> ['tak_dikenal:<nama>']. Argumen bukan dict -> ['type:<argumen>']."""
    # TODO
    raise NotImplementedError


def jalankan_tool(nama, argumen, tools=TOOLS):
    """Validasi dulu, baru eksekusi handler. Return {'hasil', 'galat', 'errors'}.
    Saat ada error: hasil None dan galat = error digabung dengan '; '."""
    # TODO
    raise NotImplementedError


In [ ]:
def loop_tool(provider, model, pesan_awal, tools=TOOLS, maks_iterasi=4):
    """Loop function calling: model minta tool -> kita eksekusi -> hasil dikirim balik.

    Berhenti karena: (a) model tidak lagi minta tool -> jawaban = teks,
    (b) `maks_iterasi` tercapai -> jawaban None + berhenti='maks_iterasi'.
    Return: jawaban, iterasi, tool_dipakai, riwayat, tokens_in, tokens_out, total_ms.
    """
    # TODO
    raise NotImplementedError


In [ ]:
# ✅ Cek latihan 6.1 — validasi argumen (model TIDAK dipercaya)
print('tools       :', [t['function']['name'] for t in skema_tools()])
print('required    :', skema_parameter('get_cuaca')['required'])
print('kurang      :', validasi_argumen('get_cuaca', {}))
print('tipe salah  :', validasi_argumen('get_cuaca', {'kota': 5, 'satuan': 'celsius'}))
print('tak dikenal :', validasi_argumen('get_cuaca', {'kota': 'a', 'satuan': 'c', 'x': 1}))
print('tool asing  :', validasi_argumen('nuklir', {}))
assert skema_parameter('get_cuaca')['required'] == ['kota', 'satuan']
assert validasi_argumen('get_cuaca', {}) == ['missing:kota', 'missing:satuan']
assert validasi_argumen('get_cuaca', {'kota': 5, 'satuan': 'celsius'}) == ['type:kota']
assert validasi_argumen('get_cuaca', {'kota': 'a', 'satuan': 'c', 'x': 1}) == ['tak_dikenal:x']
assert validasi_argumen('nuklir', {}) == ['tak_dikenal:nuklir']
assert jalankan_tool('get_cuaca', {'kota': 5})['galat'] == 'missing:satuan; type:kota'
print('✅ Latihan 6.1 benar! Argumen cacat tidak pernah sampai ke handler.')
print('   Bandingkan dengan README Bab 11 yang memakai eval(argumen[\'expression\']):')
print('   itu memberi MODEL kendali atas kode kita.')


In [ ]:
# ✅ Cek latihan 6.2 — loop tool & batas iterasi
h_tool = loop_tool(provider_utama(), 'mini',
                   'Cari catatan tentang machine learning, lalu hitung 15 * 8 + 20.')
print('tool dipakai :', h_tool['tool_dipakai'], '| iterasi', h_tool['iterasi'])
for r in h_tool['riwayat']:
    print('  iter %d -> %s(%s) = %s' % (r['iterasi'], r['tool'], r['argumen'], r['hasil']))
print('jawaban      :', h_tool['jawaban'])
assert h_tool['tool_dipakai'] == ['cari_catatan', 'hitung']
assert h_tool['iterasi'] == 3

h_satu = loop_tool(provider_utama(), 'mini', 'Berapa suhu Jakarta hari ini?')
print('satu tool    :', h_satu['tool_dipakai'], '| iterasi', h_satu['iterasi'])
assert h_satu['tool_dipakai'] == ['get_cuaca'] and h_satu['iterasi'] == 2

h_tanpa = loop_tool(provider_utama(), 'mini', 'Jelaskan machine learning singkat.')
assert h_tanpa['iterasi'] == 1 and h_tanpa['tool_dipakai'] == []

h_mentok = loop_tool(provider_utama(), 'mini', 'Berapa suhu Jakarta hari ini?', maks_iterasi=1)
print('mentok       :', h_mentok['berhenti'], '| jawaban', h_mentok['jawaban'])
assert h_mentok['berhenti'] == 'maks_iterasi' and h_mentok['jawaban'] is None
print('✅ Latihan 6.2 benar! Tool tidak pernah dipanggil dua kali dalam satu percakapan')
print('   (dicek dari assistant `tool_calls`, bukan dari teks prompt).')
print('   Agent tanpa batas iterasi = tagihan tanpa batas.')


## Bagian 7 — Orkestrasi: Pipeline yang Bisa Diaudit

Satu tugas besar dipecah menjadi langkah kecil dengan **model + suhu sendiri**.
Kegagalannya terlokalisasi, dan tiap langkah melaporkan biaya & latensinya.

```
LANGKAH_PIPELINE = [
  klasifikasi (mini,   T=0.0)
  ekstraksi   (mini,   T=0.0)
  balasan     (sedang, T=0.4)   ← satu-satunya yang butuh kreativitas
]
```


In [ ]:
def jalankan_pipeline(provider, langkah, teks, maks_percobaan=3, timeout_ms=5000):
    """Prompt chaining: jalankan langkah berurutan, laporkan tiap langkah.

    Return: langkah (detail per langkah), berhasil, gagal, total_biaya_usd,
    total_ms, teks_akhir. Langkah yang gagal biaya_usd=0.0.
    """
    # TODO
    raise NotImplementedError


In [ ]:
# ✅ Cek latihan 7.1 — laporan per langkah
pl = jalankan_pipeline(provider_utama(), LANGKAH_PIPELINE, TIKET_CONTOH)
print('tiket:', TIKET_CONTOH)
print()
print('%-12s %-7s %-7s %-9s %-7s %s' % ('langkah', 'model', 'sukses', 'attempts', 'ms', 'biaya'))
for l in pl['langkah']:
    print('%-12s %-7s %-7s %-9s %-7s $%.8f'
          % (l['nama'], l['model'], l['sukses'], l['attempts'], l['total_ms'], l['biaya_usd']))
print('total: berhasil %d | gagal %d | %d ms | $%.8f'
      % (pl['berhasil'], pl['gagal'], pl['total_ms'], pl['total_biaya_usd']))
assert (pl['berhasil'], pl['gagal'], pl['total_ms']) == (3, 0, 1620)
assert [l['attempts'] for l in pl['langkah']] == [1, 1, 2]
assert [l['total_ms'] for l in pl['langkah']] == [184, 187, 1249]
print('✅ Latihan 7.1 benar! Langkah `balasan` makan 1249 dari 1620 ms (retry + model sedang).')
print('   Tanpa laporan per langkah, kamu hanya tahu "totalnya lambat".')


In [ ]:
# ✅ Cek latihan 7.2 — anggaran percobaan & kegagalan terlokalisasi
pl_gagal = jalankan_pipeline(provider_utama(), LANGKAH_PIPELINE, TIKET_CONTOH,
                             maks_percobaan=1)
print('maks_percobaan=1 ->', [(l['nama'], l['sukses'], l['galat']) for l in pl_gagal['langkah']])
print('berhasil/gagal :', pl_gagal['berhasil'], '/', pl_gagal['gagal'])
print('biaya gagal    : $%.8f | teks_akhir: %s' % (pl_gagal['total_biaya_usd'], pl_gagal['teks_akhir']))
assert (pl_gagal['berhasil'], pl_gagal['gagal']) == (2, 1)
assert pl_gagal['langkah'][2]['galat'] == 'semua_gagal'
assert pl_gagal['langkah'][2]['biaya_usd'] == 0.0
print('✅ Latihan 7.2 benar! Percobaan gagal tidak ditagih (di mock), tapi kegagalannya')
print('   TERLAPOR — itu yang membuat alert bisa menyala di langkah yang benar.')


## Bagian 8 — Ringkasan: Satu Pekerjaan, Enam Angka

Angka-angka dari bagian sebelumnya disatukan; inilah yang dilaporkan saat
sistem ini berubah (bukan "jalan kok").


In [ ]:
# ✅ Cek latihan 8.1 — ringkasan terkunci
print('=' * 66)
print('RINGKASAN: satu pekerjaan pelanggan, biaya & latensi per komponen')
print('=' * 66)
print('1. retry    : attempts %d, backoff %s' % (h_sedang['attempts'], h_sedang['timpenungguan_ms']))
print('2. fallback : menang di tautan %d (%s)' % (h_fb['lompatan'], h_fb['model']))
print('3. cache    : hit rate %.2f -> %d panggilan provider untuk 4 permintaan'
      % (c.statistik()['hit_rate'], p_cache.jumlah_panggilan['mini']))
print('4. stream   : TTFT %d ms dari total %d ms' % (s['ttft_ms'], s['total_ms']))
print('5. tools    : %s (%d iterasi)' % (h_tool['tool_dipakai'], h_tool['iterasi']))
print('6. pipeline : %d langkah, $%.8f, %d ms'
      % (len(pl['langkah']), pl['total_biaya_usd'], pl['total_ms']))
print()
print('Enam hal yang bisa diuji TANPA jaringan, karena providernya palsu & deterministik.')
print('Itulah inti bab ini: sistem LLM yang serius diuji tepat di titik kegagalannya.')


### Koneksi ke bab lain

- **Bab 10:** template, parser, validator, guard dari bab sebelumnya — di sini
  dibungkus jadi client yang tahan gagal.
- **Bab 12 (RAG):** `konteks` yang di sini masih placeholder akan diambil dari
  dokumen nyata; **kunci cache wajib memuat konteks retrieval itu**.
- **Bab 13 (Fine-tuning):** model sendiri = satu tautan lagi di rantai fallback,
  dan harga per token jauh lebih murah (tapi kamu yang menanggung uptime-nya).
- **Bab 14 (Agent):** `loop_tool` inilah inti agent; bedanya di jumlah tool, memori,
  dan sandboxing (least-privilege).
- **Bab 15 (Evaluasi):** `attempts`, `hit_rate`, `TTFT`, biaya per langkah = metrik
  produksi yang dipantau terus-menerus.
- **Bab 16 (Deployment):** rate limit, kuota per tenant, dan anggaran (Bagian 1)
  menjadi kebijakan gateway.

**Pertanyaan analisis** (jawab di sini, benar-benar dijawab):

1. `bad_request` dilempar di percobaan ke-1, `rate_limit` diulang. Apa yang rusak
   kalau keduanya diulang dengan backoff yang sama — sebutkan dampaknya ke biaya
   DAN ke pengalaman pengguna?
2. Backoff naik 1000 → 2000 → 4000 lalu dibatasi `maks_ms=8000`. Kenapa tidak
   langsung 8000 ms sejak awal, dan kenapa batas atas itu perlu?
3. Fallback menyelamatkan permintaan, tapi apa harganya? Kapan turun ke model kecil
   justru lebih buruk daripada mengembalikan error 503?
4. Kunci cache memuat model + pesan + parameter. Sebutkan dua hal lagi yang WAJIB
   masuk kunci di produksi, dan jelaskan risiko kalau salah satunya lupa.
5. Streaming menurunkan TTFT (120 ms) dari total (340 ms). Ukuran apa yang benar-
   benar berubah, dan kapan streaming justru menambah kompleksitas tanpa manfaat?
6. Argumen tool divalidasi sebelum handler dipanggil. Jelaskan apa yang bisa terjadi
   kalau `arguments` dari model langsung dieksekusi, dan bagaimana prinsip
   least-privilege (Bab 14) membatasinya.

Lanjut ke: **kuis** (`02_kuis_llm_api.ipynb`) lalu **project starter**
(`project-llmkit/`) — paket `llmkit` end-to-end dengan **107 test**.
